# MR604 — SNN four-metric benchmark (Colab Pro)

**Before running:** Runtime → Change runtime type → **T4 GPU**, Standard RAM.

Put the `SNN-MR604` folder in your Google Drive (MyDrive) first. Then run every
cell top to bottom. Results and checkpoints go to Drive, so if Colab disconnects
you just re-run — finished configs are skipped and it resumes.

## 1. Confirm the GPU (must say Tesla T4)

In [ ]:
import torch
print('torch', torch.__version__)
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > T4 GPU'
print('GPU  :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 2. Mount Drive and enter the code folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/SNN-MR604
!ls

## 3. Install deps (torch is already on Colab) and self-check

In [ ]:
!pip install -q -r requirements.txt
!python tools/check_install.py

## 4. Smoke test — 2 epochs, 1 seed. Proves the whole pipeline before the real runs.

In [ ]:
!python run.py smoke --dataset fmnist \
  --out-dir results/_smoke --ckpt-dir checkpoints/_smoke

## 5. Full grids — 5 seeds each. Run ONE dataset per session on this machine.

Everything writes under the Drive folder (`results/<dataset>`, `checkpoints/<dataset>`),
so a disconnect is safe: re-run the same cell and it resumes.

**Do Fashion-MNIST and KMNIST here; leave one dataset for the RTX 5070 so latency
columns stay on one machine per dataset.**

### Fashion-MNIST

In [ ]:
!python run.py grid --dataset fmnist --epochs 64 \
  --seeds 42 123 789 456 1337 \
  --out-dir results/fmnist --ckpt-dir checkpoints/fmnist
!python analyse.py --out-dir results/fmnist --tables-dir tables/fmnist --figures-dir figures/fmnist

### KMNIST

In [ ]:
!python run.py grid --dataset kmnist --epochs 64 \
  --seeds 42 123 789 456 1337 \
  --out-dir results/kmnist --ckpt-dir checkpoints/kmnist
!python analyse.py --out-dir results/kmnist --tables-dir tables/kmnist --figures-dir figures/kmnist

### MNIST (only if you are NOT re-running it on the 5070)

In [ ]:
!python run.py grid --dataset mnist --epochs 64 \
  --seeds 42 123 789 456 1337 \
  --out-dir results/mnist --ckpt-dir checkpoints/mnist
!python analyse.py --out-dir results/mnist --tables-dir tables/mnist --figures-dir figures/mnist

## Notes
- **T4** is the right GPU here: the network is tiny, so A100/L4 burn units for no real speedup.
- ~2 units/hr on T4 → your 100 units cover the whole extension.
- N-MNIST / CIFAR are NOT drop-in (see CANONICAL.md) — don't add them without a code change.